# Customer Churn Prediction

## 📌 Project Overview
This project aims to predict customer churn using machine learning techniques.  
The goal is to identify customers who are likely to leave and help businesses take proactive actions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 📊 Dataset

The dataset contains customer information such as:
- Demographics
- Account details
- Services subscribed
- Churn status

> Note: Dataset should be placed in the same directory as the notebook.

In [ ]:
df=pd.read_csv('customer_churn_raw.csv')

In [ ]:
df.head()

## 🔍 Exploratory Data Analysis

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
print(df.shape)
print(df.describe())

In [ ]:
sns.countplot(x='Churn',data=df)
plt.show()

### Data Cleaning
- Removed duplicates
- Handled inconsistent values in Gender
- Treated outliers in Age

In [ ]:
df=df.drop_duplicates()
df.shape


In [ ]:
df=df.drop_duplicates(subset='CustomerID')
df.shape

In [ ]:
df.Gender.value_counts()

In [ ]:
df.columns

### * standardize inconsistent

In [ ]:
gmap={'Female':'Female',
      'F':'Female',
      'female':'Female',
      'M':'Male',
      'Male':'Male',
      'male':'Male'}
df.Gender=df.Gender.map(gmap)
df.Gender.value_counts()

In [ ]:
df.head()

In [ ]:
df['Contract_Type']=df['Contract_Type'].str.strip().str.title()
df['Internet_Service']=df['Internet_Service'].str.strip().str.title()
df[['Contract_Type','Internet_Service']].head(7)

#### handle_outliers

In [ ]:
df.loc[df['Age']>100,'Age']=np.nan

In [ ]:
df.shape

#### drop unnecessary column

In [ ]:
df=df.drop(columns=['CustomerID'])
df.shape

In [ ]:
df.head()

In [ ]:
df.isna().sum()

#### imputation methods

In [ ]:
cate_cols=['Payment_Method','Internet_Service','Region','Contract_Type']
numec_cols=['Age','Tenure_Months','Monthly_Charges','Total_Charges','Num_Products','Support_Tickets','Satisfaction_Score']

for i in cate_cols:
    df[i]=df[i].fillna(df[i].mode()[0])

for j in numec_cols:
    df[j]=df[j].fillna(df[j].median())

df.isna().sum()

#### Encode

In [ ]:
df.Churn.value_counts()

In [ ]:
df['Churn']=df['Churn'].map({'Yes':1,'No':0})
df.Churn.head()

In [ ]:
df['Gender']=df['Gender'].map({'Male':1,'Female':0})

In [ ]:
df.head()

## ⚙️ Feature Engineering
- Encoding categorical variables
- Scaling numerical features

In [ ]:

df=pd.get_dummies(df,columns=cate_cols,drop_first=True,dtype=int)

In [ ]:
df.head()

In [ ]:
df.Churn.value_counts()

##### resampling

In [ ]:
from sklearn.utils import resample

In [ ]:

maj_class=df[df['Churn']==0]
min_class=df[df['Churn']==1]
min_oversampled=resample(min_class,replace=True,random_state=42,n_samples=len(maj_class))
df_balanced=pd.concat([maj_class,min_oversampled])
print('shape :',df_balanced.shape)
print('target count :',df_balanced.Churn.value_counts())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score,classification_report,confusion_matrix,accuracy_score,recall_score,precision_score,f1_score,ConfusionMatrixDisplay 


## 🔀 Train-Test Split

In [ ]:
x=df_balanced.drop(columns='Churn')
y=df_balanced['Churn']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)


## 🤖 Model Training

In [ ]:
rf=RandomForestClassifier(n_estimators=120,random_state=42,class_weight='balanced')
rf.fit(x_train,y_train)

## 📈 Model Evaluation

In [ ]:
for name, model in [("Random Forest", rf)]:
    y_pred = model.predict(x_test)
    y_prob = model.predict_proba(x_test)[:, 1]
    print(f"\n{'─'*50}")
    print(f"  {name}")
    print(f"{'─'*50}")
    print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))
    print(f"  ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")

In [ ]:
y_pred_rd=rf.predict(x_test)
conf1=confusion_matrix(y_test,y_pred_rd)
print('\nconf for rd :\n',conf1)
ConfusionMatrixDisplay(conf1).plot(cmap='viridis')
plt.show()


In [ ]:

feature_importance = pd.Series(model.feature_importances_, index=x.columns)
feature_importance.sort_values(ascending=False).head(10).plot(kind='barh')
plt.show()

## ✅ Conclusion

- The model achieved good performance in predicting churn.
- Key factors affecting churn include:
  - Contract type
  - Monthly charges
  - Tenure
- This model can help businesses reduce customer loss by early detection.